In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

In [56]:
df = pd.read_csv("../data/raw/atp_tennis_raw.csv")

C:\Users\JumpStart\AppData\Local\Temp\ipykernel_23272\1896676511.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/atp_tennis_raw.csv")


In [57]:
def build_player_match_long(df: pd.DataFrame):
    p1 = df[["Year", "Surface", "Series", "Player_1", "Player_2", "Winner", "Round", "Rank_1", "Rank_2"]].copy()

    mask = p1["Rank_2"] < p1["Rank_1"]

# swap jugadores
    p1.loc[mask, ["Player_1", "Player_2"]] = p1.loc[mask, ["Player_2", "Player_1"]].values

# swap rankings
    p1.loc[mask, ["Rank_1", "Rank_2"]] = p1.loc[mask, ["Rank_2", "Rank_1"]].values
    p1["Win"] = (p1["Winner"] == p1["Player_1"]).astype(int)

    return p1

In [58]:
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df_test=build_player_match_long(df)

## Representation

In [42]:
def build_player_match_long(df: pd.DataFrame):
    p1 = df[["Date", "Year", "Surface", "Series", "Is_Grand_Slam", "Player_1", "Player_2", "Winner", "Round", "Rank_1", "Rank_2"]].copy()
    p1["Player"] = p1["Player_1"]
    p1["Opponent"] = p1["Player_2"]
    p1["Player_rank"] = p1["Rank_1"]
    p1["Opponent_rank"] = p1["Rank_2"]
    p1["Win"] = (p1["Winner"] == p1["Player_1"]).astype(int)

    p2 = df[["Date", "Year", "Surface", "Series", "Is_Grand_Slam", "Player_1", "Player_2", "Winner", "Round", "Rank_1", "Rank_2"]].copy()
    p2["Player"] = p2["Player_2"]
    p2["Opponent"] = p2["Player_1"]
    p2["Player_rank"] = p2["Rank_2"]
    p2["Opponent_rank"] = p2["Rank_1"]
    p2["Win"] = (p2["Winner"] == p2["Player_2"]).astype(int)

    long_df = pd.concat(
        [
            p1[["Date", "Year", "Surface", "Series", "Is_Grand_Slam", "Player", "Opponent", "Player_rank", "Opponent_rank", "Win", "Round"]],
            p2[["Date", "Year", "Surface", "Series", "Is_Grand_Slam", "Player", "Opponent", "Player_rank", "Opponent_rank", "Win", "Round"]],
        ],
        ignore_index=True,
    )

    return long_df.dropna(subset=["Player", "Opponent", "Surface", "Year", "Round"])

In [43]:
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Is_Grand_Slam"] = df["Series"].eq("Grand Slam")
data = build_player_match_long(df)

In [44]:
player_stats = (
    data.groupby("Player")
    .agg(
        Matches=("Win", "count"),
        Wins=("Win", "sum")
    )
    .reset_index()
)

player_stats["Win_rate"] = player_stats["Wins"] / player_stats["Matches"]
player_stats

,Player,Matches,Wins,Win_rate
0,Hajek J.,2,1,0.500000
1,Abdulla M.,1,0,0.000000
2,Abel M.,8,1,0.125000
3,Acasuso J.,328,171,0.521341
4,Adaktusson J.,1,0,0.000000
...,...,...,...,...
1717,di Pasquale A.,82,27,0.329268
1718,van Gemerden M.,9,3,0.333333
1719,van Lottum J.,60,18,0.300000
1720,van Scheppingen D.,43,14,0.325581


In [45]:
surface_stats = (
    data.groupby(["Player", "Surface"])
    .agg(
        Matches=("Win", "count"),
        Wins=("Win", "sum")
    )
    .reset_index()
)

surface_stats["Surface_win_rate"] = surface_stats["Wins"] / surface_stats["Matches"]
surface_stats

,Player,Surface,Matches,Wins,Surface_win_rate
0,Hajek J.,Clay,2,1,0.500000
1,Abdulla M.,Hard,1,0,0.000000
2,Abel M.,Clay,3,1,0.333333
3,Abel M.,Grass,1,0,0.000000
4,Abel M.,Hard,4,0,0.000000
...,...,...,...,...,...
3435,van Lottum J.,Hard,29,6,0.206897
3436,van Scheppingen D.,Clay,15,5,0.333333
3437,van Scheppingen D.,Grass,8,2,0.250000
3438,van Scheppingen D.,Hard,20,7,0.350000


In [46]:
data["Opponent_rank"] < data["Player_rank"]

0         False
1          True
2         False
3          True
4         False
          ...  
131307    False
131308     True
131309    False
131310    False
131311    False
Length: 131312, dtype: bool

In [47]:
data["Better_opponent"] = data["Opponent_rank"] < data["Player_rank"]

vs_better = (
    data[data["Better_opponent"]]
    .groupby("Player")
    .agg(
        Matches=("Win", "count"),
        Wins=("Win", "sum")
    )
    .reset_index()
)

vs_better["Win_vs_better"] = vs_better["Wins"] / vs_better["Matches"]
vs_better

,Player,Matches,Wins,Win_vs_better
0,Hajek J.,1,0,0.000000
1,Abdulla M.,1,0,0.000000
2,Abel M.,8,1,0.125000
3,Acasuso J.,158,67,0.424051
4,Adaktusson J.,1,0,0.000000
...,...,...,...,...
1688,di Pasquale A.,60,17,0.283333
1689,van Gemerden M.,8,2,0.250000
1690,van Lottum J.,43,12,0.279070
1691,van Scheppingen D.,31,9,0.290323


In [48]:
data["Worse_opponent"] = data["Opponent_rank"] > data["Player_rank"]

vs_worse = (
    data[data["Worse_opponent"]]
    .groupby("Player")
    .agg(
        Matches=("Win", "count"),
        Wins=("Win", "sum")
    )
    .reset_index()
)

vs_worse["Win_vs_worse"] = vs_worse["Wins"] / vs_worse["Matches"]
vs_worse

,Player,Matches,Wins,Win_vs_worse
0,Hajek J.,1,1,1.000000
1,Acasuso J.,170,104,0.611765
2,Agamenone F.,1,1,1.000000
3,Agassi A.,299,235,0.785953
4,Agenor R.,12,4,0.333333
...,...,...,...,...
933,di Mauro A.,17,6,0.352941
934,di Pasquale A.,22,10,0.454545
935,van Gemerden M.,1,1,1.000000
936,van Lottum J.,17,6,0.352941


In [60]:
round_mapping = {
    "1st Round": 1,
    "2nd Round": 2,
    "3rd Round": 3,
    "4th Round": 4,
    "Quarterfinals": 5,
    "Semifinals": 6,
    "The Final": 7
}

df_test["Round_num"] = df_test["Round"].map(round_mapping)

In [50]:
def compute_last5_form(data):

    data = data.sort_values(by="Date")

    data["Last5_win_rate"] = (
        data
        .groupby("Player")["Win"]
        .transform(lambda x: x.shift().rolling(5).mean())
    )

    return data

In [51]:
compute_last5_form(data)

,Date,Year,Surface,Series,Is_Grand_Slam,Player,Opponent,Player_rank,Opponent_rank,Win,Round,Better_opponent,Worse_opponent,Round_num,Last5_win_rate
0,2000-01-03,2000,Hard,ATP250,False,Dosedel S.,Ljubicic I.,63,77,1,1st Round,False,True,1.0,NaN
65734,2000-01-03,2000,Hard,ATP250,False,Caratti C.,Kiefer N.,211,6,0,2nd Round,True,False,2.0,NaN
65733,2000-01-03,2000,Hard,ATP250,False,El Aynaoui Y.,Gaudio G.,33,73,1,2nd Round,False,True,2.0,NaN
65732,2000-01-03,2000,Hard,ATP250,False,Berasategui A.,Bastl G.,60,85,0,2nd Round,False,True,2.0,NaN
65731,2000-01-03,2000,Hard,ATP250,False,Vacek D.,Vicente F.,39,50,1,1st Round,False,True,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65653,2026-03-14,2026,Hard,Masters 1000,False,Zverev A.,Sinner J.,4,2,0,Semifinals,True,False,6.0,0.8
131310,2026-03-14,2026,Hard,Masters 1000,False,Alcaraz C.,Medvedev D.,1,11,0,Semifinals,False,True,6.0,1.0
131309,2026-03-14,2026,Hard,Masters 1000,False,Sinner J.,Zverev A.,2,4,1,Semifinals,False,True,6.0,0.8
65655,2026-03-15,2026,Hard,Masters 1000,False,Medvedev D.,Sinner J.,11,2,0,The Final,True,False,7.0,1.0


In [52]:
tournament_results = (
    data.groupby(["Player", "Series", "Surface"])
    .agg(Max_round=("Round_num", "max"))
    .reset_index()
)
tournament_results

,Player,Series,Surface,Max_round
0,Hajek J.,ATP250,Clay,2.0
1,Abdulla M.,ATP250,Hard,1.0
2,Abel M.,ATP250,Grass,1.0
3,Abel M.,ATP250,Hard,1.0
4,Abel M.,ATP500,Clay,2.0
...,...,...,...,...
8313,van Scheppingen D.,ATP500,Hard,3.0
8314,van Scheppingen D.,Grand Slam,Clay,1.0
8315,van Scheppingen D.,Grand Slam,Grass,1.0
8316,van Scheppingen D.,Grand Slam,Hard,1.0


In [61]:
df_test

,Year,Surface,Series,Player_1,Player_2,Winner,Round,Rank_1,Rank_2,Win,Round_num
0,2000,Hard,ATP250,Dosedel S.,Ljubicic I.,Dosedel S.,1st Round,63,77,1,1.0
1,2000,Hard,ATP250,Enqvist T.,Clement A.,Enqvist T.,1st Round,5,56,1,1.0
2,2000,Hard,ATP250,Escude N.,Baccanello P.,Escude N.,1st Round,40,655,1,1.0
3,2000,Hard,ATP250,Federer R.,Knippschild J.,Federer R.,1st Round,65,87,1,1.0
4,2000,Hard,ATP250,Fromberg R.,Woodbridge T.,Fromberg R.,1st Round,81,198,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...
65651,2026,Hard,Masters 1000,Medvedev D.,Draper J.,Medvedev D.,Quarterfinals,11,14,1,5.0
65652,2026,Hard,Masters 1000,Alcaraz C.,Norrie C.,Alcaraz C.,Quarterfinals,1,29,1,5.0
65653,2026,Hard,Masters 1000,Sinner J.,Zverev A.,Sinner J.,Semifinals,2,4,1,6.0
65654,2026,Hard,Masters 1000,Alcaraz C.,Medvedev D.,Medvedev D.,Semifinals,1,11,0,6.0


In [63]:
df_test = df_test.merge(
    player_stats[["Player", "Win_rate"]].rename(columns={"Win_rate": "Win_rate_player_1"}),
    left_on="Player_1",
    right_on="Player",
    how="left"
).drop(columns=["Player"])

df_test = df_test.merge(
    player_stats[["Player", "Win_rate"]].rename(columns={"Win_rate": "Win_rate_player_2"}),
    left_on="Player_2",
    right_on="Player",
    how="left"
).drop(columns=["Player"])

In [72]:
df_test = df_test.merge(
    surface_stats[["Player", "Surface", "Surface_win_rate"]].rename(columns={"Surface_win_rate": "Surface_win_rate_player_1"}),
    left_on=["Player_1", "Surface"],
    right_on=["Player", "Surface"],
    how="left"
).drop(columns=["Player"])

df_test = df_test.merge(
    surface_stats[["Player", "Surface", "Surface_win_rate"]].rename(columns={"Surface_win_rate": "Surface_win_rate_player_2"}),
    left_on=["Player_2", "Surface"],
    right_on=["Player", "Surface"],
    how="left"
).drop(columns=["Player"])

In [76]:
df_test = df_test.merge(vs_worse[["Player", "Win_vs_worse"]].rename(columns={"Win_vs_worse": "Lose_vs_worse_player_2"}),
    left_on="Player_1",
    right_on="Player",
    how="left").drop(columns=["Player"])

df_test = df_test.merge(vs_better[["Player", "Win_vs_better"]].rename(columns={"Win_vs_better": "Win_vs_better_player_2"}),
    left_on="Player_2",
    right_on="Player",
    how="left").drop(columns=["Player"])

In [22]:
data = data.merge(player_stats[["Player", "Win_rate"]], on="Player", how="left")
data = data.merge(surface_stats[["Player", "Surface", "Surface_win_rate"]], on=["Player", "Surface"], how="left")
data = data.merge(vs_better[["Player", "Win_vs_better"]], on="Player", how="left")
data = data.merge(vs_worse[["Player", "Win_vs_worse"]], on="Player", how="left")



In [77]:
df_test

,Year,Surface,Series,Player_1,Player_2,Winner,Round,Rank_1,Rank_2,Win,Round_num,Win_rate_player_1,Win_rate_player_2,Surface_win_rate_player_1,Surface_win_rate_player_2,Player_x,Lose_vs_worse_player_2,Player_y,Win_vs_better_player_2
0,2000,Hard,ATP250,Dosedel S.,Ljubicic I.,Dosedel S.,1st Round,63,77,1,1.0,0.375000,0.594356,0.379310,0.619835,Dosedel S.,0.562500,Ljubicic I.,0.434343
1,2000,Hard,ATP250,Enqvist T.,Clement A.,Enqvist T.,1st Round,5,56,1,1.0,0.549107,0.486364,0.576923,0.500000,Enqvist T.,0.641379,Clement A.,0.401709
2,2000,Hard,ATP250,Escude N.,Baccanello P.,Escude N.,1st Round,40,655,1,1.0,0.635762,0.222222,0.680000,0.142857,Escude N.,0.666667,Baccanello P.,0.222222
3,2000,Hard,ATP250,Federer R.,Knippschild J.,Federer R.,1st Round,65,87,1,1.0,0.827252,0.341463,0.834107,0.111111,Federer R.,0.858070,Knippschild J.,0.241379
4,2000,Hard,ATP250,Fromberg R.,Woodbridge T.,Fromberg R.,1st Round,81,198,1,1.0,0.387755,0.400000,0.461538,0.272727,Fromberg R.,0.500000,Woodbridge T.,0.388889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65651,2026,Hard,Masters 1000,Medvedev D.,Draper J.,Medvedev D.,Quarterfinals,11,14,1,5.0,0.705467,0.686275,0.735714,0.723404,Medvedev D.,0.769575,Draper J.,0.565789
65652,2026,Hard,Masters 1000,Alcaraz C.,Norrie C.,Alcaraz C.,Quarterfinals,1,29,1,5.0,0.818182,0.571429,0.780220,0.565041,Alcaraz C.,0.852518,Norrie C.,0.421053
65653,2026,Hard,Masters 1000,Sinner J.,Zverev A.,Sinner J.,Semifinals,2,4,1,6.0,0.796392,0.700581,0.822642,0.692308,Sinner J.,0.874194,Zverev A.,0.391608
65654,2026,Hard,Masters 1000,Alcaraz C.,Medvedev D.,Medvedev D.,Semifinals,1,11,0,6.0,0.818182,0.705467,0.780220,0.735714,Alcaraz C.,0.852518,Medvedev D.,0.466667


In [24]:
data["Rank_diff"] = data["Player_rank"] - data["Opponent_rank"]

In [19]:
X = data[
    [
        "Round_num",
        "Win_rate",
        "Surface_win_rate",
        "Win_vs_better",
        "Rank_diff"

    ]
]

y = data["Win"]

In [20]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [21]:
def predict_match(model, player_features):

    prob = model.predict_proba(player_features)[0][1]

    return prob * 100

In [22]:
ao_2026 = data[
    (data["Series"] == "Grand Slam") &
    (data["Year"] == 2026)
]

In [23]:
predictions = []

for _, row in ao_2026.iterrows():

    features = row[X.columns]  # mismas columnas

    prob = model.predict_proba([features])[0][1]

    predictions.append(prob)

C:\Users\JumpStart\OneDrive - Hochschule Luzern\Dokumente\Lukas Lussi\Projects\Master\Semester 1\CIP\Project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\JumpStart\OneDrive - Hochschule Luzern\Dokumente\Lukas Lussi\Projects\Master\Semester 1\CIP\Project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\JumpStart\OneDrive - Hochschule Luzern\Dokumente\Lukas Lussi\Projects\Master\Semester 1\CIP\Project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\JumpStart\OneDrive - Hochschule Luzern\Dokumente\Lukas Lussi\Projects\Master\Semester 1\CIP\Project\.venv\Lib\site-pac

In [24]:
ao_2026["Predicted"] = predictions

C:\Users\JumpStart\AppData\Local\Temp\ipykernel_28556\3512888575.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ao_2026["Predicted"] = predictions


In [29]:
from joblib import dump
dump(model, "../models/atp_match_model.pkl")

['../models/atp_match_model.pkl']

In [28]:
from pathlib import Path

print(Path.cwd())

C:\Users\JumpStart\OneDrive - Hochschule Luzern\Dokumente\Lukas Lussi\Projects\Master\Semester 1\CIP\Project\.venv\app\notebooks


In [66]:
ao_2026

,Date,Year,Surface,Series,Is_Grand_Slam,Player,Opponent,Player_rank,Opponent_rank,Win,Round,Better_opponent,Round_num,Win_rate,Surface_win_rate,Win_vs_better,Rank_diff,Predicted
65158,2026-01-18,2026,Hard,Grand Slam,True,Etcheverry T.,Kecmanovic M.,62,60,1,1st Round,True,1.0,0.473684,0.433735,0.320513,2,0.772000
65159,2026-01-18,2026,Hard,Grand Slam,True,Cobolli F.,Fery A.,22,186,0,1st Round,False,1.0,0.544118,0.513514,0.377049,-164,0.170000
65160,2026-01-18,2026,Hard,Grand Slam,True,Moutet C.,Schoolkate T.,37,98,1,1st Round,False,1.0,0.443350,0.457944,0.408451,-61,0.948333
65161,2026-01-18,2026,Hard,Grand Slam,True,Diallo G.,Zverev A.,41,3,0,1st Round,True,1.0,0.472973,0.420000,0.395833,38,0.210000
65162,2026-01-18,2026,Hard,Grand Slam,True,Cerundolo F.,Zhang Zh.,21,368,1,1st Round,False,1.0,0.563025,0.489130,0.406593,-347,0.980000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130929,2026-01-27,2026,Hard,Grand Slam,True,Alcaraz C.,De Minaur A.,1,6,1,Quarterfinals,False,5.0,0.818182,0.780220,0.634615,-5,0.793167
130930,2026-01-28,2026,Hard,Grand Slam,True,Shelton B.,Sinner J.,7,2,0,Quarterfinals,True,5.0,0.598958,0.635036,0.326087,5,0.020000
130931,2026-01-30,2026,Hard,Grand Slam,True,Alcaraz C.,Zverev A.,1,3,1,Semifinals,False,6.0,0.818182,0.780220,0.634615,-2,0.630714
130932,2026-01-30,2026,Hard,Grand Slam,True,Sinner J.,Djokovic N.,2,4,0,Semifinals,False,6.0,0.796392,0.822642,0.487179,-2,0.563722


In [67]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(ao_2026["Win"], ao_2026["Predicted"])

print("Brier score:", brier)

Brier score: 0.04936219055220071


In [69]:
sample = X_test.iloc[0:1]

print("Predicción:", predict_match(model, sample))
print("Real:", y_test.iloc[0])

NameError: name 'X_test' is not defined

In [1]:
from joblib import load
model = load("../models/atp_match_model_.pkl")